# Análise de Satisfação de Clientes — Olist E-commerce

Análise exploratória dos dados públicos da Olist com foco em satisfação do cliente, tempo de entrega e desempenho por categoria de produto.

**Ferramentas:** Databricks, SQL, Delta Lake  
**Arquitetura:** Medallion (bronze → silver → gold)  
**Dados:** [Brazilian E-Commerce Public Dataset by Olist](https://www.kaggle.com/datasets/olistbr/brazilian-ecommerce)

---

## Perguntas que este projeto responde

1. Qual é a satisfação geral dos clientes da Olist?
2. Entregas mais rápidas geram notas mais altas?
3. Quais categorias de produto têm piores avaliações?
4. A satisfação variou ao longo do tempo?

## 1. Camada Bronze — ingestão dos dados brutos

Os arquivos CSV foram carregados no Databricks usando PySpark.
Um dicionário mapeia cada arquivo para o nome da tabela destino, e um loop percorre todos automaticamente — evitando repetição de código.

As tabelas bronze preservam os dados originais sem nenhuma alteração — são a fonte da verdade do projeto.

In [ ]:
arquivos = {
    "olist_customers_dataset.csv": "bronze_customers",
    "olist_orders_dataset.csv": "bronze_orders",
    "olist_order_items_dataset.csv": "bronze_order_items",
    "olist_order_payments_dataset.csv": "bronze_payments",
    "olist_order_reviews_dataset.csv": "bronze_reviews",
    "olist_products_dataset.csv": "bronze_products",
    "olist_sellers_dataset.csv": "bronze_sellers",
    "olist_geolocation_dataset.csv": "bronze_geolocation",
    "product_category_name_translation.csv": "bronze_category_translation"
}

base_path = "dbfs:/Workspace/Users/seu_usuário/Olist/"

for arquivo, tabela in arquivos.items():
    (
        spark.read
        .option("header", True)
        .option("inferSchema", True)
        .csv(base_path + arquivo)
        .write
        .mode("overwrite")
        .saveAsTable(tabela)
    )
    print(f"Criada tabela: {tabela}")

## 2. Camada Silver — limpeza dos dados

Na camada silver aplicamos as transformações necessárias:
- Conversão de colunas de texto para os tipos corretos (datas, números)
- Remoção de registros inválidos

Usamos `TRY_CAST` no lugar de `CAST` para evitar erros quando o dado está malformado — valores inválidos viram NULL e são filtrados pelo WHERE.

In [ ]:
-- Silver orders: converte datas de texto para TIMESTAMP
CREATE OR REPLACE TABLE silver_orders AS
SELECT
    order_id,
    order_status,
    customer_id,
    CAST(order_purchase_timestamp AS TIMESTAMP) AS purchase_date,
    CAST(order_delivered_customer_date AS TIMESTAMP) AS delivered_date
FROM bronze_orders;

In [ ]:
-- Silver reviews: converte review_score de texto para número inteiro
-- TRY_CAST evita erro quando encontra valores inválidos no campo
CREATE OR REPLACE TABLE silver_reviews AS
SELECT
    review_id,
    order_id,
    TRY_CAST(review_score AS INT) AS review_score,
    TRY_CAST(review_creation_date AS TIMESTAMP) AS review_date
FROM bronze_reviews
WHERE TRY_CAST(review_score AS INT) IS NOT NULL;

-- Visão geral dos reviews limpos
SELECT COUNT(*) AS total, ROUND(AVG(review_score), 2) AS nota_media
FROM silver_reviews;

**Resultado:** 99.225 reviews válidos com nota média geral de **4.09** em uma escala de 1 a 5.

---

## 3. Camada Gold — análises

Na camada gold os dados limpos são agregados para responder as perguntas do projeto.

### Análise 1 — Tempo de entrega vs satisfação

Cruzamos a nota de cada review com o tempo de entrega do pedido correspondente.
Calculamos dois indicadores:
- **media_dias_entrega:** tempo total entre a compra e a entrega
- **media_dias_atraso:** diferença entre a entrega real e a data prevista (negativo = chegou antes do prazo)

In [ ]:
CREATE OR REPLACE TABLE gold_delivery_vs_satisfaction AS
SELECT
    r.review_score,
    COUNT(*) AS total_pedidos,
    ROUND(AVG(DATEDIFF(o.delivered_date, o.purchase_date)), 1) AS media_dias_entrega,
    ROUND(AVG(DATEDIFF(
        o.delivered_date,
        TRY_CAST(ord.order_estimated_delivery_date AS TIMESTAMP)
    )), 1) AS media_dias_atraso
FROM silver_reviews r
JOIN silver_orders o ON r.order_id = o.order_id
JOIN bronze_orders ord ON r.order_id = ord.order_id
WHERE o.delivered_date IS NOT NULL
GROUP BY r.review_score
ORDER BY r.review_score;

SELECT * FROM gold_delivery_vs_satisfaction;

**Insight:** clientes com nota 5 receberam seus pedidos em média 10.6 dias e 13.4 dias antes do prazo. Clientes com nota 1 esperaram 21.3 dias e o pedido chegou apenas 4.1 dias antes do prazo — o dobro do tempo de espera.

Não é apenas sobre atrasar: chegar bem antes do prazo é o que gera nota máxima.

### Análise 2 — Satisfação por categoria de produto

Cruzamos reviews com os itens de cada pedido e os produtos correspondentes para identificar quais categorias concentram as piores avaliações.

Usamos `HAVING COUNT > 100` para garantir que só apareçam categorias com volume suficiente de reviews — evitando distorções de amostras pequenas.

In [ ]:
CREATE OR REPLACE TABLE gold_satisfaction_by_category AS
SELECT
    COALESCE(t.product_category_name_english, p.product_category_name) AS categoria,
    COUNT(r.review_id) AS total_reviews,
    ROUND(AVG(r.review_score), 2) AS nota_media
FROM silver_reviews r
JOIN bronze_order_items i ON r.order_id = i.order_id
JOIN bronze_products p ON i.product_id = p.product_id
LEFT JOIN bronze_category_translation t
    ON p.product_category_name = t.product_category_name
WHERE p.product_category_name IS NOT NULL
GROUP BY categoria
HAVING COUNT(r.review_id) > 100
ORDER BY nota_media ASC;

SELECT * FROM gold_satisfaction_by_category LIMIT 15;

**Insight:** móveis e produtos grandes lideram as piores avaliações. `office_furniture` tem nota média de 3.49 com 1.687 reviews — volume expressivo que confirma um problema estrutural, não pontual.

Produtos grandes têm maior risco de danos no transporte e atrasos logísticos, o que explica a concentração de notas baixas nessas categorias.

### Análise 3 — Evolução da satisfação ao longo do tempo

Agrupamos os reviews por mês para identificar tendências e eventos que impactaram a satisfação.

In [ ]:
CREATE OR REPLACE TABLE gold_satisfaction_over_time AS
SELECT
    DATE_FORMAT(r.review_date, 'yyyy-MM') AS periodo,
    COUNT(r.review_id) AS total_reviews,
    ROUND(AVG(r.review_score), 2) AS nota_media
FROM silver_reviews r
WHERE r.review_date IS NOT NULL
GROUP BY periodo
ORDER BY periodo ASC;

SELECT * FROM gold_satisfaction_over_time;

**Insight:** a satisfação oscila entre 3.76 e 4.34 ao longo do período. Dois momentos de queda se destacam:

- **Dezembro 2017:** nota 3.95 com 7.668 reviews — reflexo dos atrasos da Black Friday e Natal
- **Março 2018:** pior mês do período com nota 3.76 — impacto logístico do Carnaval, com pedidos feitos em fevereiro chegando atrasados

O padrão mostra que picos de volume e feriados prolongados pressionam a logística e derrubam a satisfação.

---

## Conclusão

Os dados mostram que a satisfação na Olist é fortemente influenciada pela experiência logística:

- **Velocidade de entrega** é o principal diferencial — clientes nota 5 esperam metade do tempo que clientes nota 1
- **Categorias de produtos grandes** como móveis concentram as piores avaliações por dificuldades no transporte
- **Eventos sazonais** como Carnaval e Black Friday criam picos de demanda que a logística não acompanha

Uma estratégia eficaz para melhorar a satisfação passaria por otimizar a logística de produtos grandes e reforçar a capacidade operacional nos períodos de alta demanda.